# Stellan-1 Max · HumanEval on a free Kaggle machine

Runs the same test as the repository's **Max benchmark** workflow: all 164 HumanEval problems, pass@1, greedy, each answer run against the official tests, with the app's own llama.cpp settings.

**Before running:** Settings (right-hand panel) → **Internet: On** (needs a verified phone number). Leave the accelerator on **None** (CPU).

Then **Save Version → Save & Run All (Commit)**. It runs in the background, so you can close the tab. Kaggle only shows the log and **Output** when the run ends. The run stops itself after 11 hours (Kaggle's limit is 12) and reports the problems it finished. The score is in the log's last line and in `humaneval-max.json`.

In [ ]:
%%bash
set -euo pipefail
# 1 = all 164 problems (up to about 11 hours on Kaggle's 4 cores). 4 = every 4th problem (41, about 2-3 hours).
SAMPLE=1
LLAMA_COMMIT=cd959bae99ff4bed22bef2604f45241c09bbb8c8
HUMANEVAL_COMMIT=b9453e8f1865df0a0057a1383767caa400acfbcf
HUMANEVAL_SHA256=b796127e635a67f93fb35c04f4cb03cf06f38c8072ee7cee8833d7bee06979ef
OUT=/kaggle/working/humaneval-max.json
WORK=/tmp/stick-bench; mkdir -p "$WORK"; cd "$WORK"
echo "== Machine: $(nproc) cores, $(free -g | awk '/Mem/{print $2}') GB memory, $(df -h /tmp | awk 'NR==2{print $4}') free in /tmp"

echo "== Node.js"
curl -fsSL https://nodejs.org/dist/v24.22.1/node-v24.22.1-linux-x64.tar.xz | tar -xJ
export PATH="$WORK/node-v24.22.1-linux-x64/bin:$PATH"; node --version

echo "== Stick AI's benchmark script"
rm -rf stick_ai && git clone -q --depth 1 https://github.com/alfredmurray-goat/stick_ai.git
(cd stick_ai && npm install --ignore-scripts --no-audit --no-fund --loglevel=error && npm run build --silent)

echo "== llama.cpp at the app's pinned version (about 10 minutes)"
if [ ! -x llama-build/bin/llama-server ]; then
  rm -rf llama.cpp && git init -q llama.cpp && git -C llama.cpp remote add origin https://github.com/ggml-org/llama.cpp.git
  git -C llama.cpp fetch -q --depth 1 origin "$LLAMA_COMMIT" && git -C llama.cpp checkout -q --detach FETCH_HEAD
  cmake -S llama.cpp -B llama-build -DCMAKE_BUILD_TYPE=Release -DBUILD_SHARED_LIBS=OFF -DGGML_NATIVE=OFF -DGGML_OPENMP=OFF -DGGML_METAL=OFF -DGGML_BLAS=OFF -DLLAMA_OPENSSL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF -DLLAMA_BUILD_SERVER=ON -DLLAMA_USE_PREBUILT_UI=OFF -DLLAMA_BUILD_UI=OFF > cmake.log
  cmake --build llama-build --config Release --target llama-server -j "$(nproc)" > build.log 2>&1 || { tail -30 build.log; exit 1; }
fi

echo "== Stellan-1 Max (14.3 GB, checked against its SHA-256)"
read -r url sha < <(cd stick_ai && node -e 'import("./dist/portable/max-release.js").then(m => console.log(m.maxRelease.url + " " + m.bundledMax.sha256))')
[ -f max.gguf ] || curl -fL --retry 5 --progress-bar -o max.gguf "$url"
echo "$sha  max.gguf" | sha256sum -c -

echo "== HumanEval"
curl -fsSL -o HumanEval.jsonl.gz "https://raw.githubusercontent.com/openai/human-eval/$HUMANEVAL_COMMIT/data/HumanEval.jsonl.gz"
echo "$HUMANEVAL_SHA256  HumanEval.jsonl.gz" | sha256sum -c - && gunzip -f HumanEval.jsonl.gz

echo "== Running (each line is one problem; the running score is in brackets)"
cd stick_ai
# Kaggle stops a run at 12 hours and may then keep nothing, so stop at 11 and report the problems finished so far.
timeout 11h node scripts/benchmark-humaneval.mjs --server ../llama-build/bin/llama-server --gguf ../max.gguf --model stellan-1-max --problems ../HumanEval.jsonl --out "$OUT" --sample "$SAMPLE" || echo "== Stopped at the 11-hour limit; the result below covers the problems finished."
node -e 'const d=require(process.argv[1]);const p=d.results.filter(r=>r.passed).length,n=d.results.length;const t=d.results.map(r=>r.genTokPerSec).filter(Boolean).sort((a,b)=>a-b);console.log(`\n== RESULT: ${p}/${n} (${(100*p/n).toFixed(1)}%) on ${d.cpus} CPUs (${d.cpu}), ${d.totalMemGB} GB. Median ${t[t.length>>1]?.toFixed(2)} tokens/s.`)' "$OUT"
